In [1]:
using LinearAlgebra
using DataFrames, CSV
using DrWatson

Построение платёжных матриц
A - Attakers, D - Defender

In [2]:
function build_payoff_matrices(V::Vector{Float64}, c_a::Float64, c_d::Float64)
    n = length(V)
    A = zeros(n, n)
    D = zeros(n, n)
    for i = 1:n, j = 1:n
        if i != j
            A[i, j] = V[i] - c_a
            D[i, j] = -V[i] - c_d
        else
            A[i, j] = -c_a
            D[i, j] = -c_d
        end
    end
    return A, D
end

build_payoff_matrices (generic function with 1 method)

Поиск равновесия для игры 2x2 (чистые или смешанные стратегии)

In [3]:
function mixed_nash_2x2(A::Matrix{Float64}, D::Matrix{Float64})
    for i = 1:2, j = 1:2
        if A[i, j] >= A[3-i, j] && D[i, j] >= D[i, 3-j]
            p = zeros(2);
            p[i] = 1.0
            q = zeros(2);
            q[j] = 1.0
            return (p = p, q = q, type = "pure")
        end
    end
    denomA = (A[1, 1] - A[2, 1]) - (A[1, 2] - A[2, 2])
    if abs(denomA) > 1e-10
        q1 = (A[2, 2] - A[1, 2]) / denomA
        q1 = clamp(q1, 0.0, 1.0)
    else
        q1 = 0.5
    end
    q = [q1, 1 - q1]
    denomD = (D[1, 1] - D[1, 2]) - (D[2, 1] - D[2, 2])
    if abs(denomD) > 1e-10
        p1 = (D[2, 2] - D[2, 1]) / denomD
        p1 = clamp(p1, 0.0, 1.0)
    else
        p1 = 0.5
    end
    p = [p1, 1 - p1]
    return (p = p, q = q, type = "mixed")
end

mixed_nash_2x2 (generic function with 1 method)

Одна симуляция (без сохранения в файл)

In [4]:
function run_simulation(params::Dict)
    V = params["V"]
    c_a = params["c_a"]
    c_d = params["c_d"]
    A, D = build_payoff_matrices(V, c_a, c_d)
    eq = mixed_nash_2x2(A, D)
    if eq.type == "pure"
        i = argmax(eq.p)
        j = argmax(eq.q)
        UA = A[i, j]
        UD = D[i, j]
    else
        UA = eq.p' * A * eq.q
        UD = eq.p' * D * eq.q
    end
    return Dict(
    "p_1" => eq.p[1],
    "p_2" => eq.p[2],
    "q_1" => eq.q[1],
    "q_2" => eq.q[2],
    "type" => eq.type,
    "UA" => UA,
    "UD" => UD,
    "V1" => V[1],
    "V2" => V[2],
    "c_a" => c_a,
    "c_d" => c_d,
    )
end

run_simulation (generic function with 1 method)

Генерация сетки параметров

In [5]:
function generate_params()
    dicts = []
    for v1 in [5.0, 10.0, 15.0], v2 in [5.0, 10.0, 15.0]
        for c_a in [0.0, 1.0, 3.0], c_d in [0.0, 1.0, 3.0]
            push!(dicts, Dict("V" => [v1, v2], "c_a" => c_a, "c_d" => c_d))
        end
    end
    return dicts
end

generate_params (generic function with 1 method)

Основная функция: прямой расчёт всех вариантов и сохранение CSV

In [6]:
function main_simulations()
    params_list = generate_params()
    rows = []
    for p in params_list
        res = run_simulation(p) # теперь всегда вычисляем
        push!(rows, res)
    end
    results = DataFrame(rows)
    mkpath(datadir("sims")) # убедимся, что папка существует
    CSV.write(datadir("sims", "results.csv"), results)
    return results
end

main_simulations (generic function with 1 method)

Загрузка ранее сохранённых результатов

In [7]:
function load_results()
    path = datadir("sims", "results.csv")
    if isfile(path)
        return CSV.read(path, DataFrame)
    else
        error("Файл с результатами не найден. Сначала выполните main_simulations().")
    end
end

load_results (generic function with 1 method)